## Header
**Purpose:** State the notebook purpose. For example: Load Domain 1 "FactTable1" delta table. It primarily loads/unions support domain 1 subject area fact data from the sources 1 & 2.   
**Revision History:**  *sample*
|Date |Author |Work item ID |Change short description |Notebook Runtime |Cluser config | Output Row Count | Output File Size |
|-    |-      |-            |-                        |-                |-             |-          |-                       |
|June 28, 2025 | Alias |NA |Refactored the code to improve readability & performance | 4.4 minutes | Large Cluster, 4 to 6 workers |~38.2 million |~4.1GB |
|June 29, 2025 | Alias |NA |Included delta optimize options while writing| 3 minutes |Large Cluster, 4 to 6 workers |~38.2 million |~4.1GB |


## Parameters and Configurations

In [ ]:
# Import only the requried shared functions

In [ ]:
%run ./SharedFunctions

In [ ]:
# Parameterize the file paths using config notebookes
varDBBronze = 'bronze'
varDBSilver = 'silver'
varDBGold = 'gold'
isDebug = 0

## Data Transformation

In [ ]:
# Use Spark SQL to write the logic and store the output in a dataframe. See the example below. 
# Seggregate the logics using headers for readability & easy navigation.
# Include clear comments to explain the purpose of the logic, so that other developers can easily understand and maintain it later.

### Base logic

In [ ]:
dfFactTable1 = spark.sql(f"""
  SQL Query goes here
""".format(varDBBronze, varDBSilver, varDBGold))

dfFactTable1.createOrReplaceTempView("tmpView_FactTable1")

### Derive DerviedColumn1 & DerviedColumn2

In [ ]:
dfFactTable1 = spark.sql(f"""
  SQL Query goes here
  Select 
    *,
    DerivedColumn1,
    DerivedColumn2,                                           
  from tmpView_FactTable1                                               
""".format(varDBBronze, varDBSilver, varDBGold))
dfFactTable1.createOrReplaceTempView("tmpView_FactTable1_Extended")

### Correct/Fix column data types

In [ ]:
dfFactTable1 = spark.sql("""
SELECT 
    CAST statements to correct data types
FROM tmpView_FactTable1_Extended
""")

### Write to delta table

In [ ]:
# Always write in delta format. Leverage out-of-the box spark functions to write or merge the data. Do not use any custom fucntions for writing.
# All the final write operations goes here. such as write operations w.r.t sensitive data, part files split for Azure DB.

In [ ]:
dfFactTable1.write\
  .format("Delta")\
  .option("delta.autoOptimize.optimizeWrite", "true")\
  .option("delta.autoOptimize.autoCompact", "true")\
  .mode("overwrite")\
  .saveAsTable("domain1_gold_lakehouse.FactTable1")

## Debug (Optional)

In [ ]:
# Keep the data profiling codes (optional) inside a debug conditional statement, so that they are not executed during the ETL job runs.

In [ ]:
if isDebug > 0:
  import os
  #get row count
  spark.sql("select count(*) as RowCount from domain1_gold_lakehouse.FactTable1").show()

  # get file size
  def get_directory_size(directory):
      total_size = 0
      with os.scandir(directory) as it:
          for entry in it:
              if entry.is_file():
                  total_size += entry.stat().st_size
              elif entry.is_dir():
                  total_size += get_directory_size(entry.path)
      return total_size

  # Replace ["path/to/directory1", "path/to/directory2", ...] with a list of directory paths
  directories = ["one lake file path/FactTable1"]

  for directory in directories:
      directory_size = get_directory_size(directory)
      print(f"The size of the directory at {directory} is {directory_size/ (1024 ** 3):.2f} GB")